# Hawkes 2-Point Function — Full Pipeline Demo

**Model:** Nonlinear Hawkes process, 2 populations, delta-function synaptic filters, quadratic nonlinearity

**Observable:** 2-point correlation function $\langle \delta\dot{n}_1(t_1)\, \delta\dot{n}_2(t_2) \rangle$

**Loop levels:** Tree-level ($\ell = 0$) and 1-loop ($\ell = 1$)

---

## What this notebook does

This notebook walks through the full automated Feynman diagram pipeline (Phases A–H) for a concrete MSRJD field theory.

### Section 1 — Theory / Model Information
Loads the 2-population nonlinear Hawkes model and prints all of its defining data: response fields ($\tilde{n}_i$), physical fluctuation fields ($\delta\dot{n}_i$), parameters, synaptic kernels, operators, and the nonlinear transfer function $\phi$. Then expands the MSRJD action to the required Taylor order and extracts the free-action kernel matrix $\mathbf{K}$, its Fourier transform $\hat{\mathbf{K}}(\omega)$, the retarded propagator $\hat{\mathbf{G}}(\omega) = \hat{\mathbf{K}}^{-1}(\omega)$, its poles, residue matrices, and the time-domain propagator $\mathbf{G}(t)$.

### Section 2 — Prediagram Enumeration
For the 2-point function ($k=2$), enumerates all labeled trees, undirected topologies, and directed prediagrams at tree level ($\ell=0$) and 1-loop ($\ell=1$). Plots each topology and prediagram with color-coded vertices: black = external legs, red = source vertices (noise insertions, in-degree 0), light blue = internal interaction vertices.

### Section 3 — Vertex Extraction
Reads off all interaction vertex types (`VertexType`) and noise source types (`SourceType`) from the expanded action. Each vertex type has a bigrade $(n_{\tilde{}}, n_{\text{phys}})$, a list of response and physical legs, and a coefficient. Checks that the current Taylor order is sufficient to cover the maximum vertex degree appearing in the prediagrams.

### Section 4 — Prediagram Filtering
Discards any prediagram whose internal vertex degree signatures don't match any available vertex or source type. Reports how many prediagrams survive at each loop level and plots the survivors.

### Section 5 — Type Assignment (Fully Labeled Diagrams)
Sets up the external fields for the 2-point function ($\delta\dot{n}_1$, $\delta\dot{n}_2$) and builds the field-to-matrix-index maps. Then runs the constraint-satisfaction engine on each surviving prediagram: for every valid assignment of vertex types to internal vertices and field types to edges (consistent with the propagator matrix $\hat{\mathbf{G}}$), it produces a `TypedDiagram`. Each typed diagram is printed showing its external leg assignments, vertex types with coefficients, and edge propagator labels $\hat{G}_{ij}$.

### Section 6 — Unique Diagrams & Combinatorial Factor $M(\Gamma)$
Deduplicates typed diagrams to obtain the set of unique Feynman diagrams $\Gamma$. For each unique diagram, computes the combinatorial factor $M(\Gamma)$ — the number of distinct valid leg-to-edge attachments (bijections) that realize $\Gamma$. An *attachment* assigns each leg slot of each vertex to a specific incident edge; identical legs (same field type) can be permuted freely, giving $M(\Gamma) = \prod_v \prod_{\text{groups of } k \text{ identical legs}} k!$. The diagram's contribution to the $k$-point function is $M(\Gamma) \times \prod_v \text{coeff}(v) \times \int(\text{propagators})$. Reference: Helias & Dahmen, *Statistical Field Theory for Neural Networks*, Ch. 9.

### Section 7 — Symbolic Integration (Stationary Case)
Constructs the frequency-domain integrand for representative diagrams: assigns frequency variables to edges, solves conservation constraints, and assembles the integrand as a product of propagator entries $\hat{G}_{ij}(\omega)$. Displays the unevaluated integral expression with the scalar prefactor factored out, then evaluates symbolically: tree-level diagrams reduce to algebraic expressions in the external frequency; one-loop diagrams are evaluated via the residue theorem.

In [ ]:
%display latex
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

from IPython.display import display, Math
from sage.all import (
    SR, matrix, latex, I, pi, exp, diff, solve, integrate, oo,
    dirac_delta, function, var, heaviside
)

from msrjd.core.field_theory import FieldTheory, fourier_transform, inverse_fourier_transform
from msrjd.core.vertices import extract_vertex_types, extract_source_types, available_degrees
from msrjd.enumeration.loop_diagram_enumeration import enumerate_all
from msrjd.diagrams.filter import filter_prediagrams, classify_prediagram_vertices
from msrjd.diagrams.type_assignment import (
    enumerate_typed_diagrams, enumerate_all as enumerate_all_typed,
    build_field_index_map, TypedDiagram
)
from msrjd.diagrams.causality import filter_causal
from msrjd.diagrams.symmetry import (
    combinatorial_factor, compute_all_combinatorial_factors,
    deduplicate_typed_diagrams, classify_coefficient_factors,
)
from msrjd.integration.symbolic import (
    check_propagator_available, build_integrand_stationary,
    integrate_to_time_domain, assign_frequencies, solve_conservation,
)

from models.hawkes_sage import HAWKES_MODEL

print('Imports loaded.')


---
## 1. Theory / Model Information

In [ ]:
m = HAWKES_MODEL
print(f"Model:  {m['name']}")
print(f"Convention: {m['background_rate_convention']}")
print()

print('── Index sets ──')
for name, vals in m['index_sets'].items():
    print(f'  {name}: {list(vals)}')
print()

print('── Response fields (not expanded — full integration variables) ──')
for f in m['response_fields']:
    print(f"  {f['name']}  (latex: ${f.get('latex', f['name'])}$)  — {f.get('description', '')}")
print()

print('── Physical fields (expanded around MF background) ──')
for f in m['physical_fields']:
    print(f"  {f['name']}  (latex: ${f.get('latex', f['name'])}$)  — {f.get('description', '')}")
print()

print('── Parameters ──')
for p in m.get('parameters', []):
    idx = '(indexed)' if p.get('indexed') else '(scalar)'
    dom = f", domain={p['domain']}" if p.get('domain') else ''
    print(f"  {p['name']}  {idx}{dom}  — {p.get('description', '')}")
print()

print('── Kernels ──')
for k in m.get('kernels', []):
    print(f"  {k['name']}  — {k.get('description', '')}")
print()

print('── Operators ──')
for o in m.get('operators', []):
    print(f"  {o['name']}  (latex: ${o.get('latex_name', o['name'])}$)  — {o.get('description', '')}")
print()

print('── Nonlinear functions ──')
for f in m.get('functions', []):
    print(f"  {f['name']}  (latex: ${f.get('latex', f['name'])}$)  — {f.get('description', '')}")
print()

print('── Specializations ──')
print('  φ quadratic (cubic and higher coefficients = 0)')
print('  g = δ(t)  (instantaneous synaptic coupling)')

### 1.1 Expand the action and show all sectors

In [ ]:
ft = FieldTheory(HAWKES_MODEL, taylor_order=4)
ft.expand()

R  = ft.ring()
ns = ft._ns

print(f'Taylor order: {ft.taylor_order}')
print(f'Polynomial ring: {R}')
print(f'Ring generators: {R.gens()}')
print(f'Response generators (n_tilde={ft._n_tilde}): {R.gens()[:ft._n_tilde]}')
print(f'Physical generators: {R.gens()[ft._n_tilde:]}')
print()

passed = ft.sanity_check()
print()
ft.summary()

### 1.2 Propagator

Extract the kernel matrix $\mathbf{K}$ from the free action, Fourier transform, and invert.

In [ ]:
import signal

S_free = ft.free_action()
ring_gen_names = [str(g) for g in R.gens()]

resp_names = [f'vt{i+1}' for i in ns.pop] + [f'nt{i+1}' for i in ns.pop]
phys_names = [f'dv{i+1}' for i in ns.pop] + [f'dn{i+1}' for i in ns.pop]

pos_to_row = {ring_gen_names.index(nm): i for i, nm in enumerate(resp_names)}
pos_to_col = {ring_gen_names.index(nm): j for j, nm in enumerate(phys_names)}

nf = len(resp_names)
K_data = [[SR(0)] * nf for _ in range(nf)]
for exp_vec, coeff in S_free.dict().items():
    row = col = None
    for k in range(len(ring_gen_names)):
        if exp_vec[k] > 0:
            if k in pos_to_row: row = pos_to_row[k]
            if k in pos_to_col: col = pos_to_col[k]
    if row is not None and col is not None:
        K_data[row][col] += SR(coeff)

K_mat = matrix(SR, K_data)

# Convert to kernel form
Dt       = ns.Dt
delta_D  = ns.delta_D
delta_Dp = ns.delta_Dp

def _to_kernel(c):
    c = SR(c)
    if c.has(delta_D) or c.has(ns.g):
        return c
    p0   = c.subs({Dt: 0})
    rest = (c - p0).subs({Dt: delta_Dp})
    return p0 * delta_D + rest

K_ker = matrix(SR, [[_to_kernel(K_mat[i, j]) for j in range(nf)] for i in range(nf)])

resp_sr  = [ns.vt[i] for i in ns.pop] + [ns.nt[i] for i in ns.pop]
phys_sr  = [ns.dv[i] for i in ns.pop] + [ns.dn[i] for i in ns.pop]

print('Field ordering:')
display(Math(r'\tilde{\mathbf{a}} = ' + latex(vector(resp_sr))
             + r',\qquad \mathbf{a} = ' + latex(vector(phys_sr))))
print()
print('Kernel matrix K (time domain):')
display(Math(r'\mathbf{K} = ' + latex(K_ker)))

# Fourier transform
t_var = SR.var('t')
omega = SR.var('omega', latex_name=r'\omega')

time_domain = {
    delta_D:  dirac_delta(t_var),
    delta_Dp: diff(dirac_delta(t_var), t_var),
    ns.g:     function('g')(t_var),
}

K_ft_data = [[SR(0)] * nf for _ in range(nf)]
for i in range(nf):
    for j in range(nf):
        c = K_ker[i, j]
        if not c.is_zero():
            K_ft_data[i][j] = fourier_transform(SR(c).subs(time_domain), t_var, omega)

K_ft = matrix(SR, K_ft_data)
print('Fourier-domain kernel:')
display(Math(r'\hat{\mathbf{K}}(\omega) = ' + latex(K_ft)))

# Propagator inverse
G_ft = K_ft.inverse().apply_map(lambda e: e.factor())
G_ft_explicit = True
print('Propagator:')
display(Math(r'\hat{\mathbf{G}}(\omega) = ' + latex(G_ft)))

# Adjugate, determinant, poles
adj_ft  = K_ft.adjugate()
D_omega = K_ft.det().expand()
D_prime = diff(D_omega, omega)

pole_eqs = solve(D_omega == 0, omega)
pole_vals = [eq.rhs() if hasattr(eq, 'rhs') else eq for eq in pole_eqs]

print(f'\ndet(K̂) = {D_omega}')
print(f'Poles ({len(pole_vals)}):')
for k, p in enumerate(pole_vals):
    display(Math(r'\omega_{' + str(k+1) + '} = ' + latex(p)))

# Residue matrices
C_mats = []
for omega_k in pole_vals:
    C_data = [[SR(0)] * nf for _ in range(nf)]
    for i in range(nf):
        for j in range(nf):
            n_ij = adj_ft[i, j]
            if not n_ij.is_zero():
                num = n_ij.subs({omega: omega_k})
                den = D_prime.subs({omega: omega_k})
                C_data[i][j] = (I * num / den).factor()
    C_mats.append(matrix(SR, C_data))

print('Residue matrices:')
for k, C in enumerate(C_mats):
    display(Math(r'\mathbf{C}_{' + str(k+1) + r'} = ' + latex(C)))

# Time-domain propagator
G_t = sum(C_mats[k] * exp(I * pole_vals[k] * t_var) for k in range(len(pole_vals)))
G_t = G_t.apply_map(lambda e: e.simplify_full())
print('Time-domain propagator G(t) for t > 0:')
display(Math(r'\mathbf{G}(t) = ' + latex(G_t)))

---
## 2. Prediagram Enumeration

Enumerate all trees, topologies, and prediagrams for the 2-point function at tree level ($\ell=0$) and 1-loop ($\ell=1$).

**Color code:** black = external legs, red = source vertices ($\mathrm{in}=0$), light blue = internal interaction vertices

In [ ]:
from sage.plot.plot import graphics_array

def plot_prediagrams(pds, title_prefix=''):
    """Plot prediagrams with colored vertices."""
    if not pds:
        print('  (none)')
        return
    plots = []
    for i, (D, G, leaves, internal) in enumerate(pds):
        leaves_set = set(leaves)
        color_map = {}
        for v in D.vertices():
            if v in leaves_set:
                color_map.setdefault('black', []).append(v)
            elif D.in_degree(v) == 0:
                color_map.setdefault('red', []).append(v)
            else:
                color_map.setdefault('lightblue', []).append(v)
        plots.append(D.plot(vertex_colors=color_map, vertex_size=400,
                            edge_thickness=2, title=f"{title_prefix}{i+1}"))
    n_cols = min(4, len(plots))
    n_rows = (len(plots) + n_cols - 1) // n_cols
    graphics_array(plots, n_rows, n_cols).show(figsize=[5*n_cols, 4*n_rows])

def plot_topologies(topos, title_prefix=''):
    """Plot undirected topologies."""
    if not topos:
        print('  (none)')
        return
    plots = []
    for i, (G, leaves, internal) in enumerate(topos):
        leaves_set = set(leaves)
        color_map = {}
        for v in G.vertices():
            if v in leaves_set:
                color_map.setdefault('black', []).append(v)
            else:
                color_map.setdefault('lightgray', []).append(v)
        plots.append(G.plot(vertex_colors=color_map, vertex_size=400,
                            edge_thickness=2, title=f"{title_prefix}{i+1}"))
    n_cols = min(4, len(plots))
    n_rows = (len(plots) + n_cols - 1) // n_cols
    graphics_array(plots, n_rows, n_cols).show(figsize=[5*n_cols, 4*n_rows])

print('Plot helpers defined.')

### 2.1 Tree level ($\ell = 0$)

In [ ]:
k = 2

trees_0, topos_0, pds_0, counts_0 = enumerate_all(k, 0, verbose=False)
print(f'k={k}, ell=0:  {counts_0["n_trees"]} trees,  {counts_0["n_topologies"]} topologies,  {counts_0["n_prediagrams"]} prediagrams')
print()

print('── Trees ──')
for i, (T, j, nl) in enumerate(trees_0):
    print(f'  Tree {i+1}: {T.num_verts()} vertices, {T.num_edges()} edges, {nl} leaves, j={j}')

print()
print('── Topologies ──')
plot_topologies(topos_0, title_prefix='Topo ')

print('── Prediagrams ──')
plot_prediagrams(pds_0, title_prefix='PD ')

for i, (D, G, leaves, internal) in enumerate(pds_0):
    print(f'  PD {i+1}: edges={D.edges(labels=False)}')
    for v in sorted(D.vertices()):
        role = 'leaf' if v in leaves else ('source' if D.in_degree(v)==0 else 'internal')
        print(f'    v{v}: in={D.in_degree(v)}, out={D.out_degree(v)} ({role})')

### 2.2 One loop ($\ell = 1$)

In [ ]:
trees_1, topos_1, pds_1, counts_1 = enumerate_all(k, 1, verbose=False)
print(f'k={k}, ell=1:  {counts_1["n_trees"]} trees,  {counts_1["n_topologies"]} topologies,  {counts_1["n_prediagrams"]} prediagrams')
print()

print('── Topologies ──')
plot_topologies(topos_1, title_prefix='Topo ')

print('── Prediagrams ──')
plot_prediagrams(pds_1, title_prefix='PD ')

for i, (D, G, leaves, internal) in enumerate(pds_1):
    print(f'  PD {i+1}: edges={D.edges(labels=False)}')
    for v in sorted(D.vertices()):
        role = 'leaf' if v in leaves else ('source' if D.in_degree(v)==0 else 'internal')
        print(f'    v{v}: in={D.in_degree(v)}, out={D.out_degree(v)} ({role})')

---
## 3. Vertex Extraction

Extract typed vertices (`VertexType`, `SourceType`) from the expanded action, based on the maximum vertex degree required by the prediagrams.

In [ ]:
from msrjd.enumeration.degree_scan import max_vertex_degree, scan_source_vertices

all_pds = pds_0 + pds_1
max_deg = max_vertex_degree(all_pds)
src_degs = scan_source_vertices(all_pds)
print(f'Max vertex degree across all prediagrams: {max_deg}')
print(f'Source vertex out-degrees needed: {src_degs}')
print(f'Current Taylor order: {ft.taylor_order}')
print(f'Sufficient: {ft.taylor_order >= max_deg}')
print()

vtypes = extract_vertex_types(ft)
stypes = extract_source_types(ft)

print(f'── Interaction vertex types ({len(vtypes)}) ──')
for i, vt in enumerate(vtypes):
    print(f'  V{i+1}: bigrade={vt.bigrade}, in_deg={vt.in_degree}, out_deg={vt.out_degree}')
    print(f'        resp_legs={vt.response_legs}, phys_legs={vt.physical_legs}')
    display(Math(f'\\quad\\text{{coeff}} = {latex(vt.coefficient)}'))

print(f'\n── Source types ({len(stypes)}) ──')
for i, st in enumerate(stypes):
    print(f'  S{i+1}: bigrade={st.bigrade}, out_deg={st.out_degree}')
    print(f'        resp_legs={st.response_legs}')
    display(Math(f'\\quad\\text{{coeff}} = {latex(st.coefficient)}'))

int_degs, src_degs_avail = available_degrees(vtypes, stypes)
print(f'\nAvailable interaction degrees (in, out): {sorted(int_degs)}')
print(f'Available source out-degrees: {sorted(src_degs_avail)}')

---
## 4. Prediagram Filtering

Remove prediagrams whose vertex degree signatures don't match any available vertex or source type.

In [ ]:
print('── Tree level (ell=0) ──')
kept_0, disc_0 = filter_prediagrams(pds_0, vtypes, stypes)
print(f'  {len(pds_0)} prediagrams → {len(kept_0)} kept, {disc_0} discarded')

print()
print('── 1-loop (ell=1) ──')
kept_1, disc_1 = filter_prediagrams(pds_1, vtypes, stypes)
print(f'  {len(pds_1)} prediagrams → {len(kept_1)} kept, {disc_1} discarded')

if disc_1 > 0:
    print(f'\n  Discarded prediagram indices:')
    kept_set = set(id(p) for p in kept_1)
    for i, pd in enumerate(pds_1):
        if id(pd) not in kept_set:
            D = pd[0]
            degs = [(D.in_degree(v), D.out_degree(v)) for v in pd[3]]  # internal verts
            print(f'    PD {i+1}: internal vertex degrees = {degs}')

print()
print('── Surviving prediagrams (tree level) ──')
plot_prediagrams(kept_0, title_prefix='Tree PD ')

print('── Surviving prediagrams (1-loop) ──')
plot_prediagrams(kept_1, title_prefix='1L PD ')

---
## 5. Type Assignment — Fully Labeled Diagrams

Enumerate all valid field-type assignments on each surviving prediagram.

**External legs:** $\delta\dot{n}_1$ and $\delta\dot{n}_2$ (the 2-point function fields).  
**Edge convention:** each directed edge $u \to v$ carries propagator $\hat{G}_{ij}(\omega)$ where $i$ is a response-field index from $u$ and $j$ is a physical-field index from $v$.

In [ ]:
# External fields for the 2-point function <dn1 dn2>
# dn1, dn2 are physical fields — they sit at the incoming-edge end of leaves
external_fields = [('dn', 1), ('dn', 2)]

# Build field index maps
ring_var_names_list = list(ns._ring_var_names)
n_tilde = ft._n_tilde
resp_idx, phys_idx = build_field_index_map(ring_var_names_list, n_tilde)

print('Response field index map:')
for leg, idx in sorted(resp_idx.items(), key=lambda x: x[1]):
    print(f'  {leg} → row {idx}')
print()
print('Physical field index map:')
for leg, idx in sorted(phys_idx.items(), key=lambda x: x[1]):
    print(f'  {leg} → col {idx}')

# Helper to display typed diagrams nicely
def show_typed_diagram(td, idx):
    D = td.prediagram[0]
    leaves = td.prediagram[2]
    print(f'\n{"="*60}')
    print(f'Diagram {idx}')
    print(f'{"="*60}')
    
    print('  External legs:')
    for lf, field in sorted(td.external_legs.items()):
        print(f'    leaf {lf} ← {field[0]}{field[1]}')
    
    if td.vertex_assignments:
        print('  Vertex assignments:')
        for v, vtype in sorted(td.vertex_assignments.items()):
            tname = type(vtype).__name__
            print(f'    v{v} ({tname}): bigrade={vtype.bigrade}, '
                  f'resp={vtype.response_legs}', end='')
            if hasattr(vtype, 'physical_legs'):
                print(f', phys={vtype.physical_legs}', end='')
            print()
            display(Math(f'\\quad\\text{{coeff}} = {latex(vtype.coefficient)}'))
    
    print('  Edges (propagator assignments):')
    for (u, v), (resp_leg, phys_leg) in sorted(td.edge_types.items()):
        ri, pi = td.propagator_indices.get((u, v), ('?', '?'))
        print(f'    {u} → {v}:  {resp_leg[0]}{resp_leg[1]} → {phys_leg[0]}{phys_leg[1]}  '
              f'[G_hat[{ri},{pi}]]')

### 5.1 Tree-level typed diagrams

In [ ]:
typed_tree = enumerate_all_typed(
    kept_0, external_fields, vtypes, stypes, G_ft, resp_idx, phys_idx
)
print(f'Tree-level: {len(kept_0)} prediagrams → {len(typed_tree)} typed diagrams')

for i, td in enumerate(typed_tree, 1):
    show_typed_diagram(td, f'Tree-{i}')

### 5.2 One-loop typed diagrams

In [ ]:
typed_1loop = enumerate_all_typed(
    kept_1, external_fields, vtypes, stypes, G_ft, resp_idx, phys_idx
)
print(f'1-loop: {len(kept_1)} prediagrams → {len(typed_1loop)} typed diagrams')

for i, td in enumerate(typed_1loop, 1):
    show_typed_diagram(td, f'1L-{i}')

---
## 6. Unique Diagrams & Combinatorial Factor $M(\Gamma)$

Deduplicate typed diagrams to obtain the set of unique Feynman diagrams $\Gamma$. For each, compute the **combinatorial factor**

$$M(\Gamma) = \prod_{v} \prod_{\substack{\text{groups of } k \\ \text{identical legs at } v}} k!$$

### What can be pulled outside the integral?

| Factor | Outside integral? | Condition |
|--------|------------------|-----------|
| $M(\Gamma)$ | **Always** | It's a pure integer |
| Interaction vertex $c_v$ | Only if constant | No symbols in `time_dep_params` |
| Source amplitude $c_{\text{src}}$ | Only if white noise AND constant amplitude | $\kappa = c \cdot \delta(t_1 - t_2)$ with $c$ not time-dependent |
| Source kernel $\kappa(t_1, t_2)$ | Never (for colored/general noise) | Enters integrand as $\hat{\kappa}(\omega)$ or $\kappa(t_1, t_2)$ |

### Stationarity vs factorizability

**Stationary** means all time dependencies are through time *differences* only — the system is Fourier-transformable. This is true for both white noise ($\kappa \propto \delta(\tau)$) and colored noise ($\kappa(\tau)$). But colored noise is stationary yet its kernel $\hat{\kappa}(\omega)$ **still enters the frequency integral** — it cannot be pulled out as a scalar.

### General weight formula

$$\text{weight}(\Gamma) = M(\Gamma) \times \int \prod_{v \in \text{int}} dt_v\; c_v(t_v) \times \prod_{\text{src}} \kappa(t_1, \ldots, t_k) \times \prod_{e=(u \to v)} G_{ij}(t_u, t_v)$$

The current Hawkes model has stationary white noise with constant amplitude, so all factors come outside. (Ref: Helias & Dahmen, Ch. 9.)

In [ ]:
# Get time-dependence metadata from the model
time_dep_params = HAWKES_MODEL.get('time_dependent_parameters', [])
noise_structure = HAWKES_MODEL.get('noise_structure', {
    'temporal_type': 'white', 'amplitude_params': []
})
print(f'Noise temporal type: {noise_structure.get("temporal_type", "white")}')
print(f'Noise amplitude params: {noise_structure.get("amplitude_params", [])}')
print(f'Time-dependent params: {time_dep_params if time_dep_params else "(none — stationary)"}')
print()


def show_unique_diagram(td, idx, winfo):
    """Display a unique diagram with M(Gamma) and weight structure."""
    M = winfo['M']
    print(f'\n{"="*60}')
    print(f'Unique Diagram {idx}')
    print(f'{"="*60}')

    # Combinatorial factor
    display(Math(f'M(\\Gamma) = {M}'))

    print('  External legs:')
    for lf, field in sorted(td.external_legs.items()):
        print(f'    leaf {lf} <- {field[0]}{field[1]}')

    if td.vertex_assignments:
        print('  Vertex assignments:')
        for v, vtype in sorted(td.vertex_assignments.items()):
            tname = type(vtype).__name__
            n_legs = len(vtype.response_legs)

            if tname == 'SourceType':
                sinfo = winfo['source_time_info'].get(v, {})
                nt = sinfo.get('temporal_type', 'white')
                in_int = sinfo.get('in_integrand', False)
                print(f'    v{v} (Source): bigrade={vtype.bigrade}, '
                      f'resp={vtype.response_legs}, '
                      f'noise={nt}, {n_legs} leg-time(s)'
                      f'{", IN INTEGRAND" if in_int else ", scalar"}')
            else:
                in_int = v in winfo['vertex_time_factors']
                print(f'    v{v} (Interaction): bigrade={vtype.bigrade}, '
                      f'resp={vtype.response_legs}', end='')
                if hasattr(vtype, 'physical_legs'):
                    print(f', phys={vtype.physical_legs}', end='')
                print(f', 1 vertex-time'
                      f'{", IN INTEGRAND" if in_int else ", scalar"}')

            display(Math(f'\\quad c_{{v_{v}}} = {latex(vtype.coefficient)}'))

    print('  Edges (propagator assignments):')
    for (u, v), (resp_leg, phys_leg) in sorted(td.edge_types.items()):
        ri, pi = td.propagator_indices.get((u, v), ('?', '?'))
        print(f'    {u} -> {v}:  {resp_leg[0]}{resp_leg[1]} -> {phys_leg[0]}{phys_leg[1]}  '
              f'[G_hat[{ri},{pi}]]')

    # Weight structure
    sp = winfo['scalar_prefactor']
    display(Math(
        r'\text{Scalar prefactor (outside integral): }\;'
        + latex(sp)
    ))

    if winfo['vertex_time_factors']:
        for v, td_factor in sorted(winfo['vertex_time_factors'].items()):
            display(Math(
                f'\\text{{Interaction v}}{v}'
                f'\\text{{ time-dep factor (inside integral): }}'
                f'{latex(td_factor)}'
            ))
    for v, sinfo in sorted(winfo['source_time_info'].items()):
        if sinfo['in_integrand']:
            n = sinfo['n_legs']
            t_args = ', '.join(f't_{{{v},{k+1}}}' for k in range(n))
            label = sinfo['temporal_type']
            if sinfo['amplitude_is_time_dep']:
                label += ', time-dep amplitude'
            display(Math(
                f'\\text{{Source v}}{v}'
                f'\\text{{ kernel (inside integral): }}'
                f'\\kappa({t_args})'
                f'\\quad[\\text{{{label}}}]'
            ))

    print(f'  Stationary (Fourier-transformable): '
          f'{"yes" if winfo["is_stationary"] else "no"}')


# ── Tree level ──
print('='*60)
print('TREE-LEVEL UNIQUE DIAGRAMS')
print('='*60)

unique_tree = deduplicate_typed_diagrams(typed_tree)
print(f'{len(typed_tree)} typed diagrams -> {len(unique_tree)} unique diagrams')

for i, td in enumerate(unique_tree, 1):
    winfo = classify_coefficient_factors(td, time_dep_params, noise_structure)
    show_unique_diagram(td, f'Tree-{i}', winfo)


# ── 1-loop ──
print('\n')
print('='*60)
print('1-LOOP UNIQUE DIAGRAMS')
print('='*60)

unique_1loop = deduplicate_typed_diagrams(typed_1loop)
print(f'{len(typed_1loop)} typed diagrams -> {len(unique_1loop)} unique diagrams')

for i, td in enumerate(unique_1loop, 1):
    winfo = classify_coefficient_factors(td, time_dep_params, noise_structure)
    show_unique_diagram(td, f'1L-{i}', winfo)

---
## 7. Symbolic Integration (Stationary Case)

For each unique diagram $\Gamma$, construct the **unevaluated frequency-domain
integral** that gives the time-domain contribution $C_\Gamma(t_1, t_2)$.

### Procedure

1. **Assign** a frequency variable $\omega_{e_i}$ to each directed edge.

2. **Solve conservation** at every internal vertex (interaction and source alike):
   $$\delta\!\Big(\sum_{\text{in}} \omega_e - \sum_{\text{out}} \omega_e\Big)$$
   This expresses internal edge frequencies in terms of external ones.
   If the diagram has $\ell$ loops, $\ell$ internal frequencies remain free.

3. **Build the integrand**: for each edge, substitute the resolved frequency
   into the propagator entry $\hat{G}_{i,j}(\omega_e)$.  Each external leg
   contributes an exponential $e^{\pm i\omega t}$ from the inverse Fourier
   transform (sign from leaf directionality: tail $\to e^{-i\omega t}$,
   head $\to e^{+i\omega t}$).

4. **Display** the full unevaluated integral:
   $$C_\Gamma(t_1, t_2) = M(\Gamma)\,\text{scalar\_pf}
   \;\prod_j \int\!\frac{d\omega_j}{2\pi}\;
   \Bigl[\prod_e \hat{G}_{i_e,j_e}(\omega_e)\Bigr]
   \Bigl[\prod_{\text{ext}} e^{\pm i\omega t}\Bigr]$$

The integral is over all free (loop) frequencies plus the independent
external frequencies (overall conservation reduces the number by one).


In [ ]:
# ── Propagator data dict (assembled from Section 1.2) ──
omega = SR.var('omega', latex_name=r'\omega')

propagator_data = {
    'G_ft': G_ft,
    'adj_ft': adj_ft,
    'D_omega': D_omega,
    'G_ft_explicit': True,
    'pole_vals': pole_vals,
    'C_mats': C_mats,
    'nf': nf,
}

prop_mode = check_propagator_available(propagator_data)
print(f'Propagator mode: {prop_mode}')
print(f'Poles: {pole_vals}')
print()


def show_integral(td, label, propagator_data, omega_sym):
    """
    Display the unevaluated integral and evaluated result for a typed diagram.
    """
    ir = build_integrand_stationary(
        td, propagator_data, k=2,
        omega_symbol=omega_sym,
        time_dep_params=HAWKES_MODEL.get('time_dependent_parameters', []),
        noise_structure=HAWKES_MODEL.get('noise_structure'),
    )

    prefactor = ir['scalar_prefactor']
    full_integrand = ir['full_integrand']
    ext_freqs = ir['ext_freqs']
    free_freqs = ir['free_freqs']
    ext_times = ir['ext_times']
    ell = ir['loop_number']
    overall = ir['overall_conservation']

    print(f'\n{"="*60}')
    print(f'{label}   (ell = {ell})')
    print(f'{"="*60}')

    # Diagram connectivity
    D = td.prediagram[0]
    leaves = td.prediagram[2]
    print(f'  Edges: {D.edges(labels=False)}')
    print(f'  Leaves: {leaves}')
    for v in sorted(D.vertices()):
        role = 'leaf' if v in leaves else ('source' if D.in_degree(v)==0 else 'internal')
        print(f'    v{v}: in={D.in_degree(v)}, out={D.out_degree(v)} ({role})')

    # Frequency assignments
    print(f'\n  Frequency variables:')
    for ek, omega_e in sorted(ir['edge_freqs'].items(), key=str):
        idx, u, v = ek
        print(f'    e{idx}: {u}->{v}:  {omega_e}')

    # Conservation solution
    print(f'\n  Conservation solution:')
    for ek, omega_e in sorted(ir['edge_freqs'].items(), key=str):
        resolved = omega_e.subs(ir['substitutions'])
        if resolved != omega_e:
            display(Math(f'    {latex(omega_e)} = {latex(resolved)}'))
    if overall is not None:
        display(Math(
            r'    \text{Overall conservation: }\;'
            + latex(overall) + r' = 0'
        ))

    print(f'\n  Free (loop) frequencies: {free_freqs if free_freqs else "(none)"}')
    print(f'  Independent external frequencies: {ext_freqs}')

    # ── Unevaluated integral ──
    int_vars = list(free_freqs) + list(ext_freqs)
    integrals_tex = ''
    for v in int_vars:
        integrals_tex += r'\int\!\frac{d' + latex(v) + r'}{2\pi}\;'

    pf_tex = latex(prefactor)
    integrand_tex = latex(full_integrand)

    display(Math(
        r'C_{\Gamma}(t_1, t_2) = '
        + pf_tex + r'\;'
        + integrals_tex
        + integrand_tex
    ))

    # Factored view
    integrand_only = ir['integrand']
    display(Math(
        r'\text{Propagator product: }\;'
        + latex(integrand_only)
    ))

    exp_factor = (full_integrand / integrand_only)
    try:
        exp_factor = exp_factor.simplify()
    except Exception:
        pass
    display(Math(
        r'\text{Exponential factor: }\;'
        + latex(exp_factor)
    ))

    # ── Evaluate by residues ──
    print(f'\n  Evaluating by residues...')
    try:
        td_result = integrate_to_time_domain(ir)
        result = td_result['time_domain_result']
        status = td_result['status']
        print(f'  Status: {status}')
        try:
            result = result.simplify_full()
        except Exception:
            pass
        display(Math(
            r'\boxed{C_{\Gamma}(t_1, t_2) = '
            + latex(result)
            + r'}'
        ))
    except Exception as exc:
        print(f'  Integration failed: {exc}')
        import traceback
        traceback.print_exc()

    return ir


# ═══════════════════════════════════════════════════════════════
# 7.1  Tree-level diagram
# ═══════════════════════════════════════════════════════════════
ir_tree = show_integral(unique_tree[0], 'Tree-1', propagator_data, omega)


# ═══════════════════════════════════════════════════════════════
# 7.2  1-loop diagrams (show first 3)
# ═══════════════════════════════════════════════════════════════
ir_loops = []
n_show = min(3, len(unique_1loop))
for i in range(n_show):
    ir_i = show_integral(
        unique_1loop[i], f'1-Loop-{i+1}',
        propagator_data, omega,
    )
    ir_loops.append(ir_i)


### 7.3 Numerical evaluation

Substitute concrete parameter values and numerically integrate
the frequency integrals.  Set $t_2 = 0$ and scan $\tau = t_1$
from $-50$ to $50$.

Parameters: $n^*_1 = 5$, $n^*_2 = 3$,
$w_{11} = 0.1$, $w_{12} = 0.05$, $w_{21} = 0.08$, $w_{22} = 0.06$,
$\tau_{\text{mem}} = 10$,
$\varphi'_1 = 0.5$, $\varphi'_2 = 0.4$,
$\varphi''_1 = 0.02$, $\varphi''_2 = 0.015$.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# 7.3  Numerical evaluation of integrals
# ═══════════════════════════════════════════════════════════════
import numpy as np

# Concrete parameters: quadratic phi, 2-population Hawkes
num_params = {
    SR.var('nstar1'): 5.0,
    SR.var('nstar2'): 3.0,
    SR.var('w11'): 0.1,
    SR.var('w12'): 0.05,
    SR.var('w21'): 0.08,
    SR.var('w22'): 0.06,
    SR.var('tau'): 10.0,
    SR.var('phi1_1'): 0.5,
    SR.var('phi1_2'): 0.4,
    SR.var('phi2_1'): 0.02,
    SR.var('phi2_2'): 0.015,
}

print('Numerical parameters:')
for sym, val in num_params.items():
    print(f'  {sym} = {val}')

# Check pole stability
print(f'\nPoles (numerical):')
for p_idx, p in enumerate(pole_vals):
    p_num = complex(CC(p.subs(num_params)))
    print(f'  pole {p_idx+1}: {p_num:.6f}   (Im = {p_num.imag:.4f})')


def numerical_evaluate(ir, num_params, tau_range=(-50, 50), n_tau=201):
    """
    Numerically evaluate C(tau) = C(t1, t2=0) for tau = t1.
    
    Takes the integrand_result from build_integrand_stationary,
    substitutes numerical parameters, and integrates numerically.
    """
    from sage.all import numerical_integral, CC
    
    prefactor = ir['scalar_prefactor']
    full_integrand = ir['full_integrand']
    ext_freqs = ir['ext_freqs']
    free_freqs = ir['free_freqs']
    
    # Substitute numerical parameters
    pf_num = float(prefactor.subs(num_params))
    integrand_num = full_integrand.subs(num_params)
    
    # Set t2 = 0 (stationary: C depends on tau = t1 - t2)
    t1_var = SR.var('t_1')
    t2_var = SR.var('t_2')
    integrand_tau = integrand_num.subs({t2_var: 0})
    
    # All frequency integration variables
    int_vars = list(free_freqs) + list(ext_freqs)
    n_integrals = len(int_vars)
    fourier_pf = 1.0 / (2 * float(pi))**n_integrals
    
    tau_vals = np.linspace(tau_range[0], tau_range[1], n_tau)
    C_real = np.zeros(n_tau)
    C_imag = np.zeros(n_tau)
    
    for i_tau, tau_val in enumerate(tau_vals):
        # Substitute t1 = tau_val
        f_omega = integrand_tau.subs({t1_var: float(tau_val)})
        
        # For multi-dimensional integrals (loop + external), integrate
        # sequentially.  For now handle 1D (tree) and 2D (1-loop).
        if n_integrals == 1:
            omega_v = int_vars[0]
            try:
                f_re = f_omega.real_part()
                f_im = f_omega.imag_part()
                re_val, _ = numerical_integral(f_re, -200, 200, max_points=10000)
                im_val, _ = numerical_integral(f_im, -200, 200, max_points=10000)
                C_real[i_tau] = pf_num * fourier_pf * re_val
                C_imag[i_tau] = pf_num * fourier_pf * im_val
            except Exception as e:
                C_real[i_tau] = float('nan')
                C_imag[i_tau] = float('nan')
                if i_tau == n_tau // 2:
                    print(f'  Integration failed at tau=0: {e}')
        elif n_integrals == 2:
            # Two nested 1D integrals (loop freq first, then external)
            omega_loop = int_vars[0]
            omega_ext = int_vars[1]
            try:
                # Inner integral over loop freq for each omega_ext value
                def inner_integrand_re(omega_ext_val):
                    f_inner = f_omega.subs({omega_ext: float(omega_ext_val)})
                    f_re = f_inner.real_part()
                    val, _ = numerical_integral(f_re, -200, 200, max_points=5000)
                    return val
                def inner_integrand_im(omega_ext_val):
                    f_inner = f_omega.subs({omega_ext: float(omega_ext_val)})
                    f_im = f_inner.imag_part()
                    val, _ = numerical_integral(f_im, -200, 200, max_points=5000)
                    return val
                re_val, _ = numerical_integral(inner_integrand_re, -200, 200, max_points=2000)
                im_val, _ = numerical_integral(inner_integrand_im, -200, 200, max_points=2000)
                C_real[i_tau] = pf_num * fourier_pf * re_val
                C_imag[i_tau] = pf_num * fourier_pf * im_val
            except Exception as e:
                C_real[i_tau] = float('nan')
                C_imag[i_tau] = float('nan')
                if i_tau == n_tau // 2:
                    print(f'  Integration failed at tau=0: {e}')
        else:
            print(f'  {n_integrals}-dimensional integral not yet implemented')
            break
    
    return tau_vals, C_real, C_imag


# ── Evaluate tree-level ──
print(f'\n{"="*60}')
print('Numerical evaluation: Tree-1')
print(f'{"="*60}')
tau, C_re, C_im = numerical_evaluate(ir_tree, num_params, tau_range=(-50, 50), n_tau=201)

mid = len(tau) // 2
print(f'  C(tau=0)   = {C_re[mid]:+.6e}  {C_im[mid]:+.6e}i')
print(f'  C(tau=5)   = {C_re[mid+10]:+.6e}  {C_im[mid+10]:+.6e}i')
print(f'  C(tau=-5)  = {C_re[mid-10]:+.6e}  {C_im[mid-10]:+.6e}i')
print(f'  max|Re(C)| = {np.nanmax(np.abs(C_re)):.6e}')
print(f'  max|Im(C)| = {np.nanmax(np.abs(C_im)):.6e}')

# Plot
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(tau, C_re, 'b-', linewidth=2, label='Re C(τ)')
ax1.plot(tau, C_im, 'r--', linewidth=1, label='Im C(τ)', alpha=0.7)
ax1.set_xlabel(r'$\tau = t_1 - t_2$', fontsize=13)
ax1.set_ylabel(r'$C(\tau)$', fontsize=13)
ax1.set_title('Tree-level covariance (numerical)', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)
ax1.axhline(0, color='k', linewidth=0.5)

# ── Evaluate first 1-loop diagram (if available) ──
if ir_loops:
    print(f'\n{"="*60}')
    print('Numerical evaluation: 1-Loop-1')
    print(f'{"="*60}')
    tau_l, C_re_l, C_im_l = numerical_evaluate(
        ir_loops[0], num_params, tau_range=(-50, 50), n_tau=101
    )
    mid_l = len(tau_l) // 2
    print(f'  C(tau=0)   = {C_re_l[mid_l]:+.6e}  {C_im_l[mid_l]:+.6e}i')
    print(f'  C(tau=5)   = {C_re_l[mid_l+5]:+.6e}  {C_im_l[mid_l+5]:+.6e}i')
    print(f'  max|Re(C)| = {np.nanmax(np.abs(C_re_l)):.6e}')
    print(f'  max|Im(C)| = {np.nanmax(np.abs(C_im_l)):.6e}')
    
    ax2.plot(tau_l, C_re_l, 'b-', linewidth=2, label='Re C(τ)')
    ax2.plot(tau_l, C_im_l, 'r--', linewidth=1, label='Im C(τ)', alpha=0.7)
    ax2.set_xlabel(r'$\tau = t_1 - t_2$', fontsize=13)
    ax2.set_ylabel(r'$C(\tau)$', fontsize=13)
    ax2.set_title('1-loop correction (numerical)', fontsize=14)
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    ax2.axhline(0, color='k', linewidth=0.5)
else:
    ax2.text(0.5, 0.5, 'No 1-loop diagrams', transform=ax2.transAxes,
             ha='center', va='center', fontsize=14)

plt.tight_layout()
plt.show()
